[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_13_Sequential_Dataset_Preparation_and_RNN_Prediction.ipynb)


# Experiment 13: Sequential Dataset Preparation and RNN Prediction

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 13 – Sequential Dataset Preparation and RNN Prediction

In [ ]:
import tensorflow as tf
# Synthetic hourly electricity demand with daily/weekly patterns
t=np.arange(24*365); series=100+15*np.sin(2*np.pi*t/24)+8*np.sin(2*np.pi*t/(24*7))+np.random.normal(0,3,len(t))
scaler=MinMaxScaler(); s=scaler.fit_transform(series.reshape(-1,1)).ravel()
def windows(a,lookback=24):
    X=np.array([a[i:i+lookback] for i in range(len(a)-lookback)])
    y=np.array([a[i+lookback] for i in range(len(a)-lookback)])
    return X[...,None],y
X,y=windows(s,24); cut=int(.8*len(X)); Xtr,Xte,ytr,yte=X[:cut],X[cut:],y[:cut],y[cut:]
m=tf.keras.Sequential([tf.keras.layers.Input((24,1)),tf.keras.layers.SimpleRNN(32),tf.keras.layers.Dense(1)])
m.compile("adam","mse"); m.fit(Xtr,ytr,epochs=8,batch_size=64,verbose=0)
pred=m.predict(Xte,verbose=0).ravel()
actual=scaler.inverse_transform(yte.reshape(-1,1)).ravel(); forecast=scaler.inverse_transform(pred.reshape(-1,1)).ravel()
print("MAE:",mean_absolute_error(actual,forecast),"RMSE:",np.sqrt(mean_squared_error(actual,forecast)))
plt.plot(actual[:200],label="Actual"); plt.plot(forecast[:200],label="Predicted"); plt.legend(); plt.show()